**0. Import requireed modules**

In [ ]:
!git clone https://github.com/lhusemann/ds_for_se_a1.git
!git clone https://github.com/apache/lucene.git
!cd lucene && git checkout releases/lucene/9.0.0
!pip install -U tokenizers transformers accelerate bitsandbytes

Cloning into 'ds_for_se_a1'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 121 (delta 52), reused 89 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 1.20 MiB | 4.92 MiB/s, done.
Resolving deltas: 100% (52/52), done.
Cloning into 'lucene'...
remote: Enumerating objects: 1344637, done.
remote: Counting objects: 100% (1670/1670), done.
remote: Compressing objects: 100% (1099/1099), done.
remote: Total 1344637 (delta 867), reused 722 (delta 413), pack-reused 1342967 (from 4)
Receiving objects: 100% (1344637/1344637), 521.70 MiB | 22.45 MiB/s, done.
Resolving deltas: 100% (779827/779827), done.
Updating files: 100% (7441/7441), done.
Updating files: 100% (6534/6534), done.
Note: switching to 'releases/lucene/9.0.0'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
st

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from google.colab import userdata
from pathlib import Path
import re
import json

**1. Provide the path of java files and load their source code into a list**

In [5]:
PATH_TO_LIST_WITH_RELEVANT_FILES = Path("/content/ds_for_se_a1/misc/Week3_filtering/relevant-files.json")

with open(PATH_TO_LIST_WITH_RELEVANT_FILES, "r") as f:
    relevant_files = set(json.load(f))

In [25]:
SOURCE_CODE_DIR = Path("/content/lucene/lucene/codecs")
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [28]:
def remove_license_header(content: str) -> str:
    # Removes the first /* ... */ Comment at the beginning of a file
    return re.sub(r'^\s*/\*.*?\*/\s*', '', content, count=1, flags=re.DOTALL)


files_list = []
files_names = [] # File names are unique
# .rglob() searches the root folder and all sub-folders automatically
# .glob() searches only the current directory
for file_path in SOURCE_CODE_DIR.rglob('*.java'):
    if file_path.is_file() and (file_path.name in relevant_files):
        files_names.append(file_path.name)
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            files_list.append(remove_license_header(content))

In [29]:
len(files_list)

57

**2. Retrieve the Hugging Face token securely from Colab's "Secrets" tab (the key icon on the left).**

In [ ]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

**3. Hardware optimization (Quantization)**

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # What is loaded in 4 bit? why?
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

**4. Generate embeddings**

In [ ]:
embedding_model_name = "Qodo/Qodo-Embed-1-7B"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name,token=hf_token)
model = AutoModel.from_pretrained(embedding_model_name,token=hf_token,quantization_config=bnb_config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

config.json:   0%|          | 0.00/893 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/370 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2Model(
  (embed_tokens): Embedding(151646, 3584)
  (layers): ModuleList(
    (0-27): 28 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
        (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
        (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
        (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
        (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
        (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((3584,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

**5. Construct the semantic similarity matrix**

In [ ]:
def mean_pool(token_embeddings, attention_mask):
    # Expand mask to match embedding dimensions, then zero out padding tokens
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask_expanded, dim=1) / torch.clamp(mask_expanded.sum(dim=1), min=1e-9)

def embed_source_code(code_files):
    embeddings = []
    for code in code_files:

        # Max length 512 (for current model and available memory size)
        inputs = tokenizer(code, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # Extract the embedding vector via mean pooling
        embedding = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])

        # Move to CPU and convert to numpy
        embeddings.append(embedding.cpu().numpy())

    return np.vstack(embeddings)

In [ ]:
embeddings = embed_source_code(files_list)
semantic_matrix = cosine_similarity(embeddings)

**6. Construct the structural similarity matrix**

In [30]:
# scan the filtered .rsf dependency file from week 1 to construct a matrix where each entry is the number of packages each pair of files depend on.
# executing this cell produces 'struct_matrix_raw'

SUFFIX = ".java"
file_to_files = {} # Dict of Set
filtered_rsf_path = Path("/content/ds_for_se_a1/filtered.rsf")

# Parse rsf file
with open(filtered_rsf_path, "r") as file:
    for line in file:
        parts = line.strip().split()
        if len(parts) != 3 or parts[0] != "depends":
            continue

        source, target = parts[1], parts[2]
        # Since files are needed but the rsf file lists per class, the syntax is mainclass$otherclass, so split and take first
        # Then remove package and only take the class/file name, and append file suffix
        source = source.split("$")[0].split(".")[-1] + SUFFIX
        target = target.split("$")[0].split(".")[-1] + SUFFIX

        # Self-referencing classes within a file should be ignored
        if source == target:
            continue

        if source not in file_to_files:
            file_to_files[source] = set()

        file_to_files[source].add(target)

file_to_depscount = {} # Dict of list of tuple(string <the other element>, int <count>)

for file, files in file_to_files.items():
    file_to_depscount[file] = []
    for file_2, files_2 in file_to_files.items():
        # Don't count self<->self since they are not necessarily equal across all files.
        # The next cell normalizes and THEN sets diagonal to 1, which would lead to incorrect numbers overall
        if file == file_2:
            continue

        count = len(files.intersection(files_2))

        # detect bidirectional dependencies
        if file_2 in files and file in files_2:
            count += 1

        if count == 0:
            continue

        file_to_depscount[file].append((file_2, count))

# There will be some files like org.apache.lucene.codecs.simpletext.SimpleTextUtil missing as they do not appear on the left side of the rsf
# As it has no references to other files within codecs (which is the package of interest)
# So the entries will be 0

# Use files_names to index the matrix, using data from file_to_files
size = len(files_names)
struct_matrix_raw = np.zeros((size, size))

for file, depscount in file_to_depscount.items():
    if file not in files_names:
        continue

    file_index = files_names.index(file)

    for (other, count) in depscount:
        if other not in files_names:
            continue

        other_index = files_names.index(other)
        # Matrix is symmetric by definition of the task
        struct_matrix_raw[file_index, other_index] = count
        struct_matrix_raw[other_index, file_index] = count

print(f"struct_matrix_raw Done ({size}x{size})")

struct_matrix_raw Done (57x57)


**7. Normalize the structural matrix**

In [ ]:
max_overlap = struct_matrix_raw.max()
struct_matrix = struct_matrix_raw / max_overlap if max_overlap > 0 else struct_matrix_raw
np.fill_diagonal(struct_matrix, 1.0)

**8. Combine the two matrices into one similarity matrix then apply complement.**

In [ ]:
# ALPHA dictates the weight of structural vs. semantic data.
# E.g., 0.4 means 40% Structural (RSF) and 60% Semantic (CodeBERT)
ALPHA = 0.4
combined_similarity = (ALPHA * struct_matrix) + ((1 - ALPHA) * semantic_matrix)

# Invert similarity to get distance
distance_matrix = 1.0 - combined_similarity
np.fill_diagonal(distance_matrix, 0)

**9. Apply clustering**

In [ ]:
TARGET_NUM_CLUSTERS = 30 # how such parameter could be optimized?
clusterer = AgglomerativeClustering(n_clusters=TARGET_NUM_CLUSTERS, metric='precomputed', linkage='complete') # what is 'precomputed' & 'linkage'?
clusters = clusterer.fit_predict(distance_matrix)

In [ ]:
rsf_output_path = OUTPUT_DIR / "clusters.rsf"

with open(rsf_output_path, 'w') as f:
    for file, cluster in sorted(zip(files_names, clusters), key=lambda x: x[1]):
        line = f"contain {cluster} {file}"
        print(line)
        f.write(line + "\n")

print(f"\nRSF file written to {rsf_output_path}")

contain 0 VariableGapTermsIndexWriter.java
contain 0 FixedGapTermsIndexWriter.java
contain 0 TermsIndexWriterBase.java
contain 0 FixedGapTermsIndexReader.java
contain 0 BlockTermsReader.java
contain 0 VariableGapTermsIndexReader.java
contain 0 BlockTermsWriter.java
contain 0 OrdsBlockTreeTermsWriter.java
contain 0 FSTTermsReader.java
contain 0 FSTTermsWriter.java
contain 1 TestBlockWriter.java
contain 1 TestFSTDictionary.java
contain 1 TestTermBytesComparator.java
contain 1 TestOrdsBlockTree.java
contain 1 TestSTBlockReader.java
contain 2 IntersectBlockReader.java
contain 2 FSTDictionary.java
contain 2 IndexDictionary.java
contain 3 TestVarGapDocFreqIntervalPostingsFormat.java
contain 3 TestVarGapFixedIntervalPostingsFormat.java
contain 3 TestFixedGapPostingsFormat.java
contain 3 TestBloomPostingsFormat.java
contain 3 TestFSTPostingsFormat.java
contain 3 TestDirectPostingsFormat.java
contain 4 TermBytes.java
contain 4 FSTOrdsOutputs.java
contain 4 FSTTermOutputs.java
contain 5 MurmurHa

**10. Visulizations**

In [ ]:
# According to the applied algorithm documentation, provide any visualizations for better understanding of how the clusters are formed.